# 06 - Pipeline hybride MedSigLIP + MedGemma sur dev

Objectif : developper un routage selectif dans lequel MedSigLIP reste le classifieur principal rapide et MedGemma intervient autour de ses seuils pour expliquer ou provoquer une abstention prudente.

Ce notebook ne charge jamais le split `final`. Il recherche une politique sur `dev`; toute evaluation honnete du pipeline obtenu exigera ensuite une nouvelle cohorte de patients non observee.

In [ ]:
%pip install -q "transformers>=4.56,<5" "accelerate>=1,<2" pillow pandas scikit-learn


## Configuration Kaggle

Le CSV `medsiglip_dev_predictions.csv` doit provenir du notebook 04. Dans la meme session, le chemin par defaut fonctionne. Dans une nouvelle session, ajouter les sorties du notebook 04 comme input Kaggle puis modifier uniquement `MEDSIGLIP_DEV_CSV`.

In [ ]:
from pathlib import Path, PurePosixPath
import gc
import hashlib
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

REPO_DIR = Path("/kaggle/working/ARVI-RX-DS-4B")
DATASET_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
MEDSIGLIP_DEV_CSV = Path("/kaggle/working/medsiglip_outputs/medsiglip_dev_predictions.csv")
OUTPUT_DIR = Path("/kaggle/working/hybrid_dev_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/alalab12/ARVI-RX-DS-4B.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)

if not os.environ.get("HF_TOKEN"):
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

assert torch.cuda.is_available(), "Active un accelerateur GPU Kaggle."
print(torch.cuda.get_device_name(0))


## Verifier les manifestes et charger le dev

Le prompt MedGemma et les seuils MedSigLIP restent figes. Le notebook refuse tout CSV contenant une ligne autre que `dev`.

In [ ]:
medsiglip_config = json.loads(
    (REPO_DIR / "config" / "medsiglip_zero_shot_v1.json").read_text(encoding="utf-8")
)
medgemma_config = json.loads(
    (REPO_DIR / "config" / "medgemma_baseline_v1.json").read_text(encoding="utf-8")
)
search_config = json.loads(
    (REPO_DIR / "config" / "hybrid_search_v1.json").read_text(encoding="utf-8")
)
prompt_path = REPO_DIR / "prompts" / "baseline_prompt.txt"
prompt_bytes = prompt_path.read_bytes().replace(b"\r\n", b"\n").replace(b"\r", b"\n")
assert hashlib.sha256(prompt_bytes).hexdigest() == medgemma_config["prompt_sha256_canonical_lf"]
assert search_config["development_split"] == "dev"

if not MEDSIGLIP_DEV_CSV.exists():
    candidates = list(Path("/kaggle/working").rglob("medsiglip_dev_predictions.csv"))
    raise FileNotFoundError(
        f"CSV MedSigLIP introuvable: {MEDSIGLIP_DEV_CSV}. Candidats: {candidates}"
    )

medsiglip_dev = pd.read_csv(MEDSIGLIP_DEV_CSV)
required = {"split", "expected_label", "score_opacity", "predicted_class", "image_path"}
missing = required - set(medsiglip_dev.columns)
assert not missing, f"Colonnes manquantes: {sorted(missing)}"
assert len(medsiglip_dev) == 120, f"120 lignes dev attendues, recu {len(medsiglip_dev)}"
assert medsiglip_dev["split"].eq("dev").all(), "Le notebook hybride refuse tout split autre que dev."
assert medsiglip_dev["technical_error"].fillna("").eq("").all()

if "case_id" not in medsiglip_dev:
    medsiglip_dev["case_id"] = [f"hybrid_dev_{index:03d}" for index in range(len(medsiglip_dev))]
assert medsiglip_dev["case_id"].is_unique
medsiglip_dev = medsiglip_dev.rename(columns={
    "predicted_class": "medsiglip_predicted_class",
    "latency_ms": "medsiglip_latency_ms",
})
display(pd.crosstab(medsiglip_dev["expected_label"], medsiglip_dev["medsiglip_predicted_class"]))


In [ ]:
def resolve_image_path(raw_path):
    direct = Path(str(raw_path))
    if direct.exists():
        return direct
    parts = PurePosixPath(str(raw_path).replace("\\", "/")).parts
    if parts and parts[0] == "CheXpert-v1.0-small":
        parts = parts[1:]
    return DATASET_ROOT.joinpath(*parts)

medsiglip_dev["image_path_resolved"] = medsiglip_dev["image_path"].map(resolve_image_path)
assert medsiglip_dev["image_path_resolved"].map(Path.exists).all()

LOW_THRESHOLD = float(medsiglip_config["low_threshold"])
HIGH_THRESHOLD = float(medsiglip_config["high_threshold"])
MARGINS = [float(value) for value in search_config["candidate_margins"]]
MAX_MARGIN = max(MARGINS)
route_pool_mask = medsiglip_dev["score_opacity"].between(
    max(0.0, LOW_THRESHOLD - MAX_MARGIN),
    min(1.0, HIGH_THRESHOLD + MAX_MARGIN),
    inclusive="both",
)
route_pool = medsiglip_dev.loc[route_pool_mask].copy()
print(f"Pool MedGemma maximal: {len(route_pool)}/120 images")
display(pd.crosstab(route_pool["expected_label"], route_pool["medsiglip_predicted_class"]))


## Preparer MedGemma

Si MedSigLIP est encore charge dans cette session, ses objets sont liberes avant MedGemma. Le smoke test utilise une image `dev`, jamais une image finale.

In [ ]:
globals().pop("model", None)
globals().pop("processor", None)
gc.collect()
torch.cuda.empty_cache()

os.environ["MODEL_BACKEND"] = "medgemma"
os.environ["MEDGEMMA_MODEL_ID"] = medgemma_config["model_id"]
os.environ["MEDGEMMA_MAX_NEW_TOKENS"] = str(medgemma_config["max_new_tokens"])
sys.path.insert(0, str(REPO_DIR))

from src.hybrid import agreement_or_abstain, routing_bounds
from src.inference import get_medgemma_backend, predict

get_medgemma_backend.cache_clear()
RUN_MEDGEMMA_SMOKE = False
if RUN_MEDGEMMA_SMOKE:
    smoke_prediction = predict(
        route_pool.iloc[0]["image_path_resolved"],
        mode=medgemma_config["mode"],
        backend="medgemma",
    )
    display(smoke_prediction)
else:
    print("Passer RUN_MEDGEMMA_SMOKE a True avant le lancement complet.")


## Inferer le pool route avec checkpoint

Passer `RUN_MEDGEMMA_DEV` a `True` apres un smoke test valide. Le CSV est reecrit apres chaque image et les predictions valides sont reprises automatiquement apres une interruption.

In [ ]:
RUN_MEDGEMMA_DEV = False
ROUTED_CSV = OUTPUT_DIR / "medgemma_routed_dev_predictions.csv"

if RUN_MEDGEMMA_DEV:
    if ROUTED_CSV.exists():
        checkpoint = pd.read_csv(ROUTED_CSV)
        checkpoint = checkpoint.drop_duplicates("case_id", keep="last")
        records = checkpoint.to_dict(orient="records")
        completed = set(
            checkpoint.loc[checkpoint["technical_error"].fillna("").eq(""), "case_id"]
        )
    else:
        records = []
        completed = set()

    pending = route_pool.loc[~route_pool["case_id"].isin(completed)]
    for position, (_, row) in enumerate(pending.iterrows(), start=1):
        record = {"case_id": row["case_id"]}
        try:
            prediction = predict(
                row["image_path_resolved"],
                mode=medgemma_config["mode"],
                backend="medgemma",
            )
            record.update({
                "medgemma_predicted_class": prediction["predicted_class"],
                "medgemma_confidence": prediction.get("confidence"),
                "medgemma_image_quality": prediction.get("image_quality"),
                "medgemma_latency_ms": prediction.get("latency_ms"),
                "medgemma_prediction_json": json.dumps(prediction),
                "technical_error": "",
            })
        except Exception as error:
            record.update({
                "medgemma_predicted_class": "technical_error",
                "medgemma_confidence": np.nan,
                "medgemma_image_quality": "",
                "medgemma_latency_ms": np.nan,
                "medgemma_prediction_json": "",
                "technical_error": str(error),
            })
        records.append(record)
        checkpoint = pd.DataFrame(records).drop_duplicates("case_id", keep="last")
        checkpoint.to_csv(ROUTED_CSV, index=False)
        print(f"{position}/{len(pending)} - {row['case_id']}")
else:
    print("Passer RUN_MEDGEMMA_DEV a True apres le smoke test.")


## Rechercher la politique de routage

Regle `agreement_or_abstain` : un accord est conserve; une abstention MedSigLIP peut etre resolue par une classe MedGemma definitive; tout autre desaccord devient `uncertain`. Une erreur MedGemma conserve la sortie primaire et reste comptee comme erreur technique.

In [ ]:
assert ROUTED_CSV.exists(), "Executer d'abord le pool MedGemma route."
routed = pd.read_csv(ROUTED_CSV).drop_duplicates("case_id", keep="last")
missing_cases = set(route_pool["case_id"]) - set(routed["case_id"])
assert not missing_cases, f"Predictions MedGemma manquantes: {sorted(missing_cases)}"
error_rows = routed["technical_error"].fillna("").ne("")
if error_rows.any():
    display(routed.loc[error_rows, ["case_id", "technical_error"]])
    raise RuntimeError("Corriger puis relancer les erreurs techniques avant la recherche.")

dev_joined = medsiglip_dev.merge(routed, on="case_id", how="left")

def summarize_policy(name, data, routed_mask):
    definitive = data[data["expected_label"].isin(["normal", "suspected_opacity"])].copy()
    opacity = definitive["expected_label"].eq("suspected_opacity")
    normal = definitive["expected_label"].eq("normal")
    routed_latency = data.loc[routed_mask, "medgemma_latency_ms"].fillna(0).sum()
    medsiglip_mean = data["medsiglip_latency_ms"].mean()
    return {
        "policy": name,
        "strict_accuracy": definitive["hybrid_predicted_class"].eq(definitive["expected_label"]).mean(),
        "opacity_sensitivity": definitive.loc[opacity, "hybrid_predicted_class"].eq("suspected_opacity").mean(),
        "opacity_to_normal_rate": definitive.loc[opacity, "hybrid_predicted_class"].eq("normal").mean(),
        "review_sensitivity": definitive.loc[opacity, "hybrid_predicted_class"].ne("normal").mean(),
        "normal_specificity": definitive.loc[normal, "hybrid_predicted_class"].eq("normal").mean(),
        "normal_to_opacity_rate": definitive.loc[normal, "hybrid_predicted_class"].eq("suspected_opacity").mean(),
        "uncertain_rate": definitive["hybrid_predicted_class"].eq("uncertain").mean(),
        "routing_rate": routed_mask.mean(),
        "estimated_mean_latency_ms": medsiglip_mean + routed_latency / len(data),
    }

policy_rows = []
policy_predictions = {}
primary = dev_joined.copy()
primary["hybrid_predicted_class"] = primary["medsiglip_predicted_class"]
no_routing = pd.Series(False, index=primary.index)
policy_rows.append(summarize_policy("medsiglip_only", primary, no_routing))
policy_predictions["medsiglip_only"] = primary

for margin in MARGINS:
    candidate = dev_joined.copy()
    route_low, route_high = routing_bounds(LOW_THRESHOLD, HIGH_THRESHOLD, margin)
    routed_mask = candidate["score_opacity"].between(route_low, route_high, inclusive="both")
    candidate["hybrid_predicted_class"] = candidate["medsiglip_predicted_class"]
    candidate.loc[routed_mask, "hybrid_predicted_class"] = candidate.loc[routed_mask].apply(
        lambda row: agreement_or_abstain(
            row["medsiglip_predicted_class"], row["medgemma_predicted_class"]
        ),
        axis=1,
    )
    policy_name = f"margin_{margin:.3f}"
    policy_rows.append(summarize_policy(policy_name, candidate, routed_mask))
    policy_predictions[policy_name] = candidate

policy_search = pd.DataFrame(policy_rows)
display(policy_search)


In [ ]:
constraints = search_config["selection_constraints"]
primary_metrics = policy_search.loc[policy_search["policy"].eq("medsiglip_only")].iloc[0]
eligible = policy_search.loc[
    policy_search["policy"].ne("medsiglip_only")
    & policy_search["opacity_to_normal_rate"].le(primary_metrics["opacity_to_normal_rate"] + 1e-12)
    & policy_search["normal_specificity"].ge(constraints["min_normal_specificity"])
    & policy_search["uncertain_rate"].le(constraints["max_uncertain_rate"])
    & policy_search["routing_rate"].le(constraints["max_routing_rate"])
].copy()
if eligible.empty:
    selection_status = "no_hybrid_candidate_met_constraints"
    selected = primary_metrics
    print("Aucune politique hybride admissible: conservation de MedSigLIP seul.")
else:
    selection_status = "hybrid_candidate_selected_on_dev"
    eligible = eligible.sort_values(
        ["strict_accuracy", "opacity_to_normal_rate", "normal_specificity", "routing_rate"],
        ascending=[False, True, False, True],
    )
    selected = eligible.iloc[0]
selected_name = selected["policy"]
selected_predictions = policy_predictions[selected_name].copy()
selected_predictions["image_path_resolved"] = selected_predictions["image_path_resolved"].astype(str)

candidate_manifest = {
    "version": "hybrid_dev_candidate_v1",
    "source_search": search_config["version"],
    "selected_policy": selected_name,
    "selected_margin": (
        None if selected_name == "medsiglip_only"
        else float(selected_name.removeprefix("margin_"))
    ),
    "metrics_dev": {key: float(selected[key]) for key in policy_search.columns if key != "policy"},
    "selection_status": selection_status,
    "final_split_used": False,
    "validation_status": "development_only_requires_new_untouched_cohort",
}
policy_search.to_csv(OUTPUT_DIR / "hybrid_dev_policy_search.csv", index=False)
selected_predictions.to_csv(OUTPUT_DIR / "hybrid_dev_candidate_predictions.csv", index=False)
with (OUTPUT_DIR / "hybrid_dev_candidate.json").open("w", encoding="utf-8") as file:
    json.dump(candidate_manifest, file, indent=2)

display(pd.DataFrame([primary_metrics, selected]))
display(pd.crosstab(selected_predictions["expected_label"], selected_predictions["hybrid_predicted_class"]))
print(candidate_manifest)


## Revue des desaccords

La revue porte sur les cas routes et ne sert pas a modifier manuellement les sorties. Elle documente les modes d'echec et la valeur ajoutee reelle de MedGemma.

In [ ]:
changed = selected_predictions.loc[
    selected_predictions["hybrid_predicted_class"].ne(
        selected_predictions["medsiglip_predicted_class"]
    )
].copy()
review_columns = [
    "case_id", "patient_id", "expected_label", "score_opacity",
    "medsiglip_predicted_class", "medgemma_predicted_class",
    "hybrid_predicted_class", "medgemma_confidence",
    "medgemma_image_quality", "image_path",
]
review_columns = [column for column in review_columns if column in changed.columns]
display(changed[review_columns])
print("Cohorte CheXpert -1, sans accuracy:")
display(
    selected_predictions.loc[
        selected_predictions["expected_label"].eq("uncertain"),
        "hybrid_predicted_class",
    ].value_counts(normalize=True, dropna=False)
)


## Decision apres execution

1. Ne jamais appliquer retrospectivement la politique candidate aux 30 cas finaux pour la recalibrer.
2. Examiner les erreurs et la couverture sur `dev`.
3. Figer la politique seulement si elle apporte un gain utile sous les contraintes.
4. Constituer une nouvelle cohorte patient-disjointe pour l'evaluation du pipeline hybride.
5. Conserver MedSigLIP seul si le routage n'apporte pas de gain robuste.